In [ ]:
import BioSimSpace as BSS
from dask.distributed import Client, LocalCluster, wait

from pathlib import Path
from uuid import uuid4

In [ ]:
config = {}
# read config
with open("output_setup/protocol.dat", "r") as file:
    for line in file:
        key, value = line.split("=")
        config[str(key).strip()] = str(value).strip().replace("*", "")

In [ ]:
# our solvation node will expect box edges to be a float, which will then be assumed to be in nanometers
# so we need to format it correctly here
config["box edges"] = BSS.Types.Length(config["box edges"]).nanometers().value()

In [ ]:
# set node directory
BSS.Node.setNodeDirectory("nodes")

In [ ]:
cluster = LocalCluster(
    n_workers=4,
    memory_limit="5000MB",
    threads_per_worker=5,
    # We will use CPU_task to ensure that only a single parameterisation/solvation is performed at a time
    # and GPU to ensure that only a single simulation is run per worker.
    resources={"CPU_task": 1, "GPU": 1},
)

In [ ]:
client = Client(cluster)
# node_plugin = SetNodeDir(node_path)
# client.register_plugin(node_plugin, name="node-setup")
client

In [ ]:
paramed_files = Path("inputs/ligands").glob("*.sdf")
files = []
for file in paramed_files:
    files.append(str(Path.cwd() / str(file)))

In [ ]:
# To ensure that our nodes know where to look, we will need to use absolute paths
protein_files = [
    str(Path("inputs/protein.prm7").absolute()),
    str(Path("inputs/protein.rst7").absolute()),
]

In [ ]:
my_inputs = []
# now we make a node-compatible input for each sdf file
# these will be used to queue our tasks
# this will also generate a unique file_prefix for each process
# negating any file conflict issues
for file in files:
    my_inputs.append(
        {
            "file": file,
            "protein files": protein_files,
            "ligand forcefield": config["ligand forcefield"],
            "water model": config["solvent"],
            "box length": config["box edges"],
            "output suffix": "solv",
            "file_prefix": str(uuid4()),
        }
    )

In [ ]:
node_path = "./nodes"
# We run this to make sure all of our workers have the correct node directory
f1 = client.run(BSS.Node.setNodeDirectory, node_path)

In [ ]:
futures = [
    client.submit(BSS.Node.run, "param_solvate", inp, resources={"CPU_task": 1})
    for inp in my_inputs
]
_ = wait(futures)
# these futures just return filenames, so we can pull all of their results back to the main thread
results_param_solvate = client.gather(futures)

In [ ]:
# now we create our inputs for the next stage of the workflow
# for the sake of simplicity we will do free and bound systems separately
next_stage_inputs_free = []
next_stage_inputs_bound = []
# we need to make sure we're using absolute paths, otherwise dask may fail to find our files
for i in results_param_solvate:
    b = i["bound solvated"]
    new_b = [str(Path(j).resolve()) for j in b]
    next_stage_inputs_bound.append(new_b)

    f = i["free solvated"]
    new_f = [str(Path(k).resolve()) for k in f]
    next_stage_inputs_free.append(new_f)
client.cancel(futures)

In [ ]:
# We will now move to GPU-bound tasks, which will require us to scale our cluster down such that the number of
# workers is equal to the number of available GPUs (in this case 1)
# now we can scale
cluster.scale(1)

In [ ]:
# first minimise the free legs
# make the input dictionaries
minimisation_inputs_free = []
for inp in next_stage_inputs_free:
    minimisation_inputs_free.append({"file": inp, "file_prefix": str(uuid4())})

In [ ]:
futures_min_free = [
    client.submit(BSS.Node.run, "minimisation", inp, resources={"GPU": 1})
    for inp in minimisation_inputs_free
]
_ = wait(futures_min_free)
result_minimisation_free = client.gather(futures_min_free)

In [ ]:
# Since we've saved our minimised systems in files, we can get rid of the futures to save memory
client.cancel(futures_min_free)

In [ ]:
# now the bound systems
minimisation_inputs_bound = []
for inp in next_stage_inputs_bound:
    minimisation_inputs_bound.append({"file": inp, "file_prefix": str(uuid4())})

In [ ]:
futures_min_bound = [
    client.submit(BSS.Node.run, "minimisation", inp, resources={"GPU": 1})
    for inp in minimisation_inputs_bound
]
_ = wait(futures_min_bound)
result_minimisation_bound = client.gather(futures_min_bound)

In [ ]:
client.cancel(futures_min_bound)

In [ ]:
# Now we will equilibrate the free legs using the free leg equilibration node
# first make input dictionaries
equilibration_inputs_free = []
for s in result_minimisation_free:
    print(Path(s["minimised"][0]).resolve())
    equilibration_inputs_free.append(
        {"file": [str(Path(i)) for i in s], "file_prefix": str(uuid4())}
    )

In [ ]:
# now equilibrate
futures_eq_free = [
    client.submit(BSS.Node.run, "equilibration_free", inp, resources={"GPU": 1})
    for inp in equilibration_inputs_free
]
_ = wait(futures_eq_free)
free_eq_systems = client.gather(futures_eq_free)

In [ ]:
client.cancel(futures_eq_free)

In [ ]:
equilibration_inputs_bound = []
for s in result_minimisation_bound:
    equilibration_inputs_bound.append(
        {"file": [str(Path(i)) for i in s], "file_prefix": str(uuid4())}
    )

In [ ]:
futures_eq_bound = [
    client.submit(BSS.Node.run, "equilibration_bound", inp, resources={"GPU": 1})
    for inp in equilibration_inputs_bound
]
_ = wait(futures_eq_bound)
bound_eq_systems = client.gather(futures_eq_bound)

In [ ]:
client.cancel(futures_eq_bound)

In [ ]:
# close the client and cluster